# 01. Данные и обучающая выборка (ML-1, ML-2, ML-3)

Ноутбук для экспериментов. Код, который должен остаться, живёт в модулях, а ноутбук только вызывает его:
- `tns/ml/proto_panel.py` — временная панель в формате `docs/interfaces.md`, пока нет панели из `tns/data` (DATA-4);
- `tns/ml/dataset.py` — срезы, признаки, таргет, проверки на утечку.

**Договорённости**
- Срез M = состояние на 01.M. Признаки: `start`-колонки месяца M и всё за месяцы ≤ M−1, окна до 6 мес.
- Выборка: только должники, `dz ≥ P-01` и `dz_overdue > 0`.
- Таргет `y` = `pay30_m`: оплачено (файл 03) ≥ 30 % `dz` до конца M.
- Деление: train 2025-07…10 · valid 2025-11 · test 2025-12…2026-02 · **holdout 2026-03 — не трогаем до ML-7**.

In [1]:
%load_ext autoreload
%autoreload 2

import logging
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "CLAUDE.md").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from tns.ml import dataset as ds
from tns.ml import proto_panel as pp

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s", force=True)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 80)

DATA_DIR = ROOT / "Материалы к кейсу"
CACHE_DIR = ROOT / "output" / "cache" / "proto"
OUT_DIR = ROOT / "output" / "ml"
RULES = ROOT / "docs" / "Правила воздействия ТНС v0.2.xlsx"

MIN_DZ = ds.min_debt(RULES)  # P-01
print("P-01 =", MIN_DZ)

P-01 = 500.0


## 1. Прото-панель

Первый запуск читает файлы 01–13 (~30 с), дальше всё берётся из кэша `output/cache/proto/`.
`refresh=True` — пересобрать принудительно (например, после правок в `proto_panel.py`).

In [2]:
tables = pp.load_proto_tables(DATA_DIR, CACHE_DIR, refresh=False)
panel, events, ls_info, payments = (tables[k] for k in ["panel", "measure_events", "ls_info", "payments"])

for name, t in tables.items():
    print(f"{name:15s} {t.shape}")
print("месяцы панели:", panel["month"].min().date(), "…", panel["month"].max().date())

INFO Прото-панель: читаю кэш из /Users/aleksejsuharev/Desktop/hackathon_tns_energo/output/cache/proto


ls_info         (150787, 9)
panel           (2261805, 15)
measure_events  (290350, 6)
payments        (1565019, 3)
месяцы панели: 2025-01-01 … 2026-03-01


In [3]:
# Сверка: оплаты из ОСВ и из файла 03 по месяцам (DATA-8). Расхождения — в риски.
chk = panel.assign(match=(panel["paid"] - panel["paid_payments"]).abs() <= 1)
chk.groupby("month")["match"].mean().round(3).to_frame("доля ЛС, где оплаты 03 и ОСВ совпадают ±1 ₽").T

month,2025-01-01,2025-02-01,2025-03-01,2025-04-01,2025-05-01,2025-06-01,2025-07-01,2025-08-01,2025-09-01,2025-10-01,2025-11-01,2025-12-01,2026-01-01,2026-02-01,2026-03-01
"доля ЛС, где оплаты 03 и ОСВ совпадают ±1 ₽",0.998,0.999,0.999,0.998,0.998,0.998,0.998,0.998,0.998,0.991,0.998,0.999,0.999,0.999,0.991


In [4]:
# Сальдо сходится между месяцами? saldo(M+1) = saldo(M) + начислено − оплачено.
# Последний месяц (2026-03) — это второй блок «2026-02» в ОСВ.
p = panel.sort_values(["ls", "month"])
nxt = p.groupby("ls")["saldo_start"].shift(-1)
ok = ((nxt - (p["saldo_start"] + p["accrual"] - p["paid"])).abs() <= 1).astype(float).where(nxt.notna())
ok.groupby(p["month"]).mean().round(3).to_frame("сальдо сходится со следующим месяцем").T

month,2025-01-01,2025-02-01,2025-03-01,2025-04-01,2025-05-01,2025-06-01,2025-07-01,2025-08-01,2025-09-01,2025-10-01,2025-11-01,2025-12-01,2026-01-01,2026-02-01,2026-03-01
сальдо сходится со следующим месяцем,0.982,0.977,0.96,0.984,0.981,0.976,0.975,0.974,0.971,0.967,0.975,0.97,0.979,0.978,NaN


## 2. Кого считать должником

Сравниваем `dz ≥ P-01` с фильтром по просрочке и без него. Если в ЛС без просрочки почти все платят,
это обычные плательщики со свежим счётом: в выборке они только завышали бы AUC.

In [5]:
cur = panel[panel["dz"] >= MIN_DZ].assign(has_overdue=lambda d: d["dz_overdue"] > 0)
pop = (
    cur.groupby(["month", "has_overdue"])
    .agg(rows=("ls", "size"), pay30=("pay30_m", "mean"))
    .unstack("has_overdue")
)
pop.columns = [f"{a}_{'просрочка' if b else 'без_просрочки'}" for a, b in pop.columns]
pop.round(3)

,rows_без_просрочки,rows_просрочка,pay30_без_просрочки,pay30_просрочка
month,,,,
2025-01-01,44387.0,NaN,0.737,<NA>
2025-02-01,29671.0,17443.0,0.927,0.457
2025-03-01,29818.0,17404.0,0.934,0.475
2025-04-01,26582.0,16726.0,0.919,0.446
2025-05-01,30352.0,16997.0,0.896,0.434
2025-06-01,31577.0,18785.0,0.919,0.439
2025-07-01,26742.0,18820.0,0.92,0.459
2025-08-01,32989.0,18103.0,0.915,0.47
2025-09-01,35255.0,18421.0,0.923,0.474


In [6]:
# Насколько велика просроченная часть у тех, кого берём. Копеечная просрочка — кандидат на доп. порог.
debt = cur[cur["has_overdue"]]
print("доля с dz_overdue < 100 ₽:", round((debt["dz_overdue"] < 100).mean(), 3))
print("доля с цензурированным сроком (долг старше 2025-01):", round(debt["overdue_censored"].mean(), 3))
debt["overdue_m"].value_counts().sort_index().to_frame("ЛС-месяцев по сроку просрочки").T

доля с dz_overdue < 100 ₽: 0.038
доля с цензурированным сроком (долг старше 2025-01): 0.363


overdue_m,0,1,2,3,4,5,6,7,8,9,10,11,12,13
ЛС-месяцев по сроку просрочки,17443,93647,44540,22095,14375,10703,8495,7318,6327,5650,5245,4266,3978,3467


## 3. Срезы и таргет

In [7]:
df = ds.build_dataset(tables, MIN_DZ)
FEATURES = ds.feature_columns(df)
print(df.shape, "признаков:", len(FEATURES))
ds.summary(df)

INFO Срез 2025-07: 18820 строк
INFO Срез 2025-08: 18103 строк
INFO Срез 2025-09: 18421 строк
INFO Срез 2025-10: 17906 строк
INFO Срез 2025-11: 16982 строк
INFO Срез 2025-12: 18049 строк
INFO Срез 2026-01: 16659 строк
INFO Срез 2026-02: 17746 строк
INFO Срез 2026-03: 17508 строк


(160194, 64) признаков: 57


,split,rows,y,y_overdue,y_any,dz_median
month,,,,,,
2025-07-01,train,18820,0.459,0.472,0.506,1643.400
2025-08-01,train,18103,0.470,0.481,0.511,1723.970
2025-09-01,train,18421,0.474,0.487,0.520,1698.910
2025-10-01,train,17906,0.485,0.498,0.532,1737.165
2025-11-01,valid,16982,0.476,0.490,0.524,1804.680
2025-12-01,test,18049,0.502,0.514,0.548,1709.990
2026-01-01,test,16659,0.435,0.449,0.480,1732.480
2026-02-01,test,17746,0.474,0.488,0.522,1840.630
2026-03-01,holdout,17508,0.477,0.492,0.530,1850.245


In [8]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
df.to_parquet(OUT_DIR / "snapshots.parquet", index=False)
print("сохранено:", OUT_DIR / "snapshots.parquet")

сохранено: /Users/aleksejsuharev/Desktop/hackathon_tns_energo/output/ml/snapshots.parquet


In [9]:
# Пропуски по признакам: months_since_* = NaN значит «меры не было за 6 мес.».
na = df[FEATURES].isna().mean().round(3)
na[na > 0].sort_values(ascending=False).to_frame("доля NaN").T

,months_since_restriction,months_since_claim,months_since_court_result,months_since_visit,months_since_court_application,months_since_email,months_since_operator_call,months_since_sms,months_since_autodial,months_since_restriction_notice,months_since_any_measure,pay30_lag3,pay_ratio_3m,pay30_lag2,pay_ratio_6m,dz_to_norm,house_type
доля NaN,0.965,0.944,0.929,0.915,0.91,0.837,0.83,0.783,0.563,0.519,0.294,0.08,0.063,0.052,0.047,0.035,0.019


## 4. Проверки на утечку (ML-3)

- `check_no_leakage`: признаки среза не меняются, если удалить всё, что стало известно после 01.M.
- `check_panel_start_cols`: `start`-колонки панели (срок просрочки, норматив, месяцы без оплат) не зависят от будущего.

Позже переедут в `tests/test_ml_leakage.py`.

In [10]:
for m in ds.ALL_MONTHS:
    ds.check_no_leakage(tables, m, MIN_DZ)
print("признаки: утечки нет во всех срезах")

for m in ["2025-07", "2025-11", "2026-03"]:
    ds.check_panel_start_cols(tables, pd.Timestamp(m))
print("start-колонки панели: утечки нет")

признаки: утечки нет во всех срезах
start-колонки панели: утечки нет


## 5. Первый взгляд на признаки

AUC каждого признака по отдельности на valid (2025-11). Значение < 0.5 — признак работает «наоборот»
(например, чем больше месяцев без оплат, тем меньше шанс оплаты), важно расстояние от 0.5.

In [11]:
from sklearn.metrics import roc_auc_score

valid = df[df["split"] == "valid"]
rows = []
for c in FEATURES:
    if c == "house_type":
        continue
    x = valid[c]
    m = x.notna()
    if m.sum() > 100 and valid.loc[m, "y"].nunique() == 2:
        rows.append((c, roc_auc_score(valid.loc[m, "y"], x[m]), 1 - m.mean()))
uni = pd.DataFrame(rows, columns=["feature", "auc", "share_nan"])
uni["strength"] = (uni["auc"] - 0.5).abs()
uni.sort_values("strength", ascending=False).head(20).round(3)

,feature,auc,share_nan,strength
27,pay30_rate_6m,0.863,0.000,0.363
22,n_pay_months_6m,0.840,0.000,0.340
4,overdue_m,0.168,0.000,0.332
9,dz_to_norm,0.170,0.036,0.330
3,overdue_share,0.174,0.000,0.326
6,months_no_payment,0.209,0.000,0.291
2,dz_overdue,0.222,0.000,0.278
19,paid_sum_6m,0.765,0.000,0.265
18,paid_sum_3m,0.755,0.000,0.255
25,pay30_lag2,0.754,0.043,0.254


In [12]:
# Доля оплативших по сроку просрочки и по оплате в прошлом месяце (train).
train = df[df["split"] == "train"]
display(train.groupby(train["overdue_m"].clip(upper=8))["y"].agg(["mean", "size"]).round(3).T)
display(train.groupby(train["pay30_lag1"])["y"].agg(["mean", "size"]).round(3).T)

overdue_m,1.0,2.0,3.0,4.0,5.0,6.0,7.0,8.0
mean,0.776,0.649,0.453,0.34,0.139,0.104,0.075,0.062
size,25137.000,13114.000,6085.000,3605.00,8347.000,6508.000,5607.000,4847.000


pay30_lag1,0.0,1.0
mean,0.387,0.815
size,58723.000,14514.000


## 6. Черновой LightGBM — проверка на здравый смысл

Это не ML-6: без подбора параметров, монотонности и калибровки. Задача — убедиться, что AUC в разумных
пределах (ожидаем 0.75–0.85; > 0.95 — ищем утечку).

In [13]:
import lightgbm as lgb

def xy(part):
    d = df[df["split"] == part]
    return d[FEATURES], d["y"]

X_tr, y_tr = xy("train")
X_va, y_va = xy("valid")

model = lgb.LGBMClassifier(n_estimators=400, learning_rate=0.05, num_leaves=31, min_child_samples=100,
                           subsample=0.8, subsample_freq=1, colsample_bytree=0.8, verbose=-1)
model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], eval_metric="auc",
          callbacks=[lgb.early_stopping(50, verbose=False)])

print("деревьев:", model.best_iteration_)
print("AUC train:", round(roc_auc_score(y_tr, model.predict_proba(X_tr)[:, 1]), 4))
print("AUC valid:", round(roc_auc_score(y_va, model.predict_proba(X_va)[:, 1]), 4))

деревьев: 169
AUC train: 0.8998
AUC valid: 0.8924


In [14]:
imp = pd.Series(model.booster_.feature_importance("gain"), index=FEATURES)
(imp / imp.sum()).sort_values(ascending=False).head(20).round(3).to_frame("доля gain").T

,pay30_rate_6m,overdue_m,dz_to_norm,overdue_share,n_pay_months_6m,pay_ratio_6m,dz_change_1m,overdue_censored,months_no_payment_censored,house_type,paid_lag1_to_dz,dz_change_3m,pay_ratio_3m,dz_max_6m,accrual_lag1,accrual_mean_6m,months_no_payment,paid_sum_3m,pay30_lag2,dz
доля gain,0.588,0.096,0.084,0.025,0.021,0.019,0.017,0.012,0.01,0.009,0.009,0.008,0.008,0.006,0.005,0.005,0.005,0.005,0.005,0.005


In [15]:
# Без анкеты ls_info (снимок на дату выгрузки, для старых срезов слегка «из будущего»).
no_info = [c for c in FEATURES if c not in ds.LS_INFO_FLAGS + ["house_type"]]
m2 = lgb.LGBMClassifier(**model.get_params()).set_params(n_estimators=model.best_iteration_)
m2.fit(X_tr[no_info], y_tr)
print("AUC valid без ls_info:", round(roc_auc_score(y_va, m2.predict_proba(X_va[no_info])[:, 1]), 4))

AUC valid без ls_info: 0.892


## 7. Кто получил меру в месяце среза

Таргет «заплатил сам» на истории означает «заплатил при той практике, что была»: часть должников в течение M
получала СМС, звонки, уведомления. Смотрим, сколько таких строк и как у них с оплатой.

Это **только для анализа**: мера в M становится известна после 01.M, поэтому как признак её брать нельзя.

In [16]:
# Меры месяца M у строк выборки (end-информация, только для анализа).
MEASURES = list(events["measure"].cat.categories)
got = (
    events[["ls", "month", "measure"]].drop_duplicates().assign(v=1)
    .pivot_table(index=["ls", "month"], columns="measure", values="v", aggfunc="max", observed=False, fill_value=0)
    .reset_index()
)
treat = df[["ls", "month", "split", "y"]].merge(got, on=["ls", "month"], how="left")
treat[MEASURES] = treat[MEASURES].fillna(0)
treat["any_measure"] = (treat[MEASURES].sum(axis=1) > 0).astype(int)

cols = MEASURES + ["any_measure"]
pd.DataFrame({
    "доля получивших в M": treat[cols].mean(),
    "y у получивших": [treat.loc[treat[c] == 1, "y"].mean() for c in cols],
    "y у не получивших": [treat.loc[treat[c] == 0, "y"].mean() for c in cols],
}).round(3)

,доля получивших в M,y у получивших,y у не получивших
autodial,0.316,0.362,0.524
email,0.123,0.460,0.474
sms,0.147,0.473,0.473
operator_call,0.061,0.357,0.480
claim,0.024,0.056,0.483
visit,0.029,0.242,0.480
restriction_notice,0.223,0.211,0.548
restriction,0.009,0.602,0.471
court_application,0.016,0.043,0.480
court_result,0.011,0.061,0.477


In [17]:
print("доля строк с мерой в M по частям:")
display(treat.groupby("split")["any_measure"].mean().round(3).to_frame().T)
print("из оплативших (y = 1) меру в M получили:", round(treat.loc[treat["y"] == 1, "any_measure"].mean(), 3))

# Как модель из раздела 6 работает на тех, кто получил меру в M, и на остальных (valid).
va = valid.assign(p=model.predict_proba(valid[FEATURES])[:, 1]).merge(
    treat[["ls", "month", "any_measure"]], on=["ls", "month"])
pd.DataFrame([
    {"мера в M": bool(g), "строк": len(d), "AUC": roc_auc_score(d["y"], d["p"]),
     "факт y": d["y"].mean(), "средний p": d["p"].mean()}
    for g, d in va.groupby("any_measure")
]).round(3)

доля строк с мерой в M по частям:


split,holdout,test,train,valid
any_measure,0.402,0.578,0.502,0.593


из оплативших (y = 1) меру в M получили: 0.376


,мера в M,строк,AUC,факт y,средний p
0,False,6912,0.954,0.663,0.554
1,True,10070,0.837,0.347,0.406


**Выводы (прогон 05.11.2026)**
- **53 % строк** получили хотя бы одну меру в месяце среза. Чаще всего это автодозвон (32 %) и уведомление (22 %).
  Значит, таргет на истории — это «оплата при старой практике», а не «оплата без мер». Так и формулируем в
  описании модели и на конференции.
- У получивших меру доля оплаты ниже (0,34 против 0,62). Это **не эффект меры**, а отбор: меры давали тем,
  кто хуже платит. Причинных выводов отсюда делать нельзя (та же ловушка, что в хакатоне).
- Модель хуже ранжирует тех, кто получил меру (AUC 0,84 против 0,95), и систематически ошибается в среднем:
  у не получивших недооценивает оплату (p 0,55 при факте 0,66), у получивших переоценивает (0,41 при 0,35).
  Калибровку (ML-6) стоит проверять по этим двум группам отдельно.
- В holdout (2026-03) меру получили только 40 % против 58–59 % в valid и test. Возможно, журналы мер за март
  неполные (риск D6). Перед ML-7 проверить.

## 8. AUC внутри пула текущей практики

AUC 0,89 посчитан по всем должникам, где легко отличить «опаздывающих» от «хронических». Аллокатору же нужно
упорядочить **кандидатов**. Берём пул, как его описал заказчик:
- долг > P-15 (1 000 ₽);
- срок просрочки P-16 (1–3 мес.);
- нет введённого ограничения. Прокси: не было ограничения за окно признаков, так как даты снятия в данных нет.

Сравниваем, как разные способы ранжирования упорядочивают пул по вероятности оплаты.

In [18]:
import re

P15 = float(ds.load_rule_param(RULES, "P-15"))
P16 = str(ds.load_rule_param(RULES, "P-16"))
lo, hi = map(int, re.findall(r"\d+", P16)[:2])  # «1–3 мес. (реже до 6)» → 1, 3
print(f"P-15 = {P15:.0f} ₽, P-16 = «{P16}» → срок {lo}–{hi} мес.")

in_pool = (va["dz"] > P15) & va["overdue_m"].between(lo, hi) & (va["n_restriction_6m"] == 0)
pool = va[in_pool]
print(f"пул: {len(pool)} из {len(va)} должников valid, сами оплатили ≥ 30 %: {pool['y'].mean():.3f}")

def auc(d, score):
    """AUC по строкам, где score не пустой."""
    m = score.notna()
    return roc_auc_score(d.loc[m, "y"], score[m])

pd.DataFrame([
    (name, auc(pool, f(pool)), auc(va, f(va)))
    for name, f in [
        ("модель (p_pay)", lambda d: d["p"]),
        ("по долгу (текущая практика)", lambda d: -d["dz"]),
        ("pay30_rate_6m", lambda d: d["pay30_rate_6m"]),
        ("короче просрочка", lambda d: -d["overdue_m"]),
    ]
], columns=["ранжирование", "AUC в пуле", "AUC все должники"]).round(3)

P-15 = 1000 ₽, P-16 = «1–3 мес. (реже до 6)» → срок 1–3 мес.
пул: 6305 из 16982 должников valid, сами оплатили ≥ 30 %: 0.706


,ранжирование,AUC в пуле,AUC все должники
0,модель (p_pay),0.779,0.892
1,по долгу (текущая практика),0.494,0.706
2,pay30_rate_6m,0.737,0.863
3,короче просрочка,0.629,0.832


**Выводы (прогон 05.11.2026)**
- **В пуле текущей практики 71 % должников оплачивают ≥ 30 % сами.** Почти три из четырёх мер уходят тем,
  кто заплатил бы и так. Это главный аргумент за ML-приоритет.
- Внутри пула AUC модели **0,78**: ожидаемый диапазон 0,75–0,85, и это честная цифра для конференции.
  0,89 по всем должникам завышена тем, что «хронических» отличить легко.
- **Сумма долга внутри пула ничего не говорит об оплате (AUC 0,49 ≈ монетка).** Сортировка «по убыванию
  долга» по вероятности оплаты работает как случайный порядок. Выиграет ли ML у неё по захвату неоплаченного
  долга, покажет бэктест (ML-4): там важен и размер долга, а не только вероятность.

## 9. Новые ЛС

Работает ли модель на ЛС, которых не было в train? Это ответ на вопрос «модель запомнила людей или выучила
поведение».

In [19]:
seen = set(df.loc[df["split"] == "train", "ls"])
va["в_train"] = va["ls"].isin(seen)
pd.DataFrame([
    {"ЛС был в train": g, "строк": len(d), "AUC": roc_auc_score(d["y"], d["p"]), "факт y": d["y"].mean()}
    for g, d in va.groupby("в_train")
]).round(3)

,ЛС был в train,строк,AUC,факт y
0,False,2462,0.811,0.793
1,True,14520,0.887,0.422


**Выводы (прогон 05.11.2026)**
- На новых ЛС AUC **0,81** против 0,89 на знакомых. Это меньше, но модель не просто запомнила людей.
- Новые должники — другая группа: 79 % из них платят сами (против 42 %). Это в основном «свежие» опоздавшие,
  у которых короче история, поэтому часть разницы в AUC объясняется составом, а не переобучением.

## 10. Окно истории: 3 месяца или 6

С окном 3 мес. можно начать срезы с апреля 2025 и получить на ~52 тыс. строк обучения больше, но признаки
видят меньше истории. Три варианта, все проверяются на одном valid (2025-11):
- окно 6, train 07–10 — текущий вариант;
- окно 3, train 07–10 — те же строки, только короче история: эффект **окна**;
- окно 3, train 04–10 — короче история, но больше строк: эффект **объёма**.

In [20]:
LGB_PARAMS = dict(n_estimators=400, learning_rate=0.05, num_leaves=31, min_child_samples=100,
                  subsample=0.8, subsample_freq=1, colsample_bytree=0.8, verbose=-1)

def fit_auc(train, valid, features):
    m = lgb.LGBMClassifier(**LGB_PARAMS)
    m.fit(train[features], train["y"], eval_set=[(valid[features], valid["y"])], eval_metric="auc",
          callbacks=[lgb.early_stopping(50, verbose=False)])
    return roc_auc_score(valid["y"], m.predict_proba(valid[features])[:, 1])

early = [pd.Timestamp(m) for m in ["2025-04", "2025-05", "2025-06"]]
df3 = ds.build_dataset(tables, MIN_DZ, months=early + ds.ALL_MONTHS, window=3)
df3["split"] = df3["split"].fillna("train_early")
F3 = ds.feature_columns(df3)
tr3, va3 = df3[df3["split"] == "train"], df3[df3["split"] == "valid"]
tr3_long = df3[df3["split"].isin(["train", "train_early"])]
tr6 = df[df["split"] == "train"]

pd.DataFrame([
    ("окно 6, train 07–10", len(tr6), fit_auc(tr6, valid, FEATURES)),
    ("окно 3, train 07–10", len(tr3), fit_auc(tr3, va3, F3)),
    ("окно 3, train 04–10", len(tr3_long), fit_auc(tr3_long, va3, F3)),
], columns=["вариант", "строк train", "AUC valid"]).round(4)

INFO Срез 2025-04: 16726 строк
INFO Срез 2025-05: 16997 строк
INFO Срез 2025-06: 18785 строк
INFO Срез 2025-07: 18820 строк
INFO Срез 2025-08: 18103 строк
INFO Срез 2025-09: 18421 строк
INFO Срез 2025-10: 17906 строк
INFO Срез 2025-11: 16982 строк
INFO Срез 2025-12: 18049 строк
INFO Срез 2026-01: 16659 строк
INFO Срез 2026-02: 17746 строк
INFO Срез 2026-03: 17508 строк


,вариант,строк train,AUC valid
0,"окно 6, train 07–10",73250,0.8924
1,"окно 3, train 07–10",73250,0.8881
2,"окно 3, train 04–10",125758,0.8882


**Выводы (прогон 05.11.2026)**
- Окно 6 мес. немного лучше: 0,892 против 0,888.
- Дополнительные срезы апреля–июня (+52 тыс. строк) ничего не добавляют: 0,8882 против 0,8881.
- **Оставляем окно 6 мес. и срезы с июля.** Объём данных не ограничивает модель, а длинная история полезна.

## Что дальше

- **ML-4** (`02_backtest.ipynb`): бэктест очереди на test-месяцах. Пул = как в разделе 8; лимиты из файла 14;
  очереди «по долгу» / `dz × (1 − p)` / идеальная; метрика — доля неоплаченного долга, попавшая в лимит.
  Отдельно считать по тем, кто меру в M не получал (раздел 7).
- **ML-5 / ML-6**: скоринговая карта и LightGBM с монотонностью + калибровка, в том числе по группам
  «получил меру в M / нет».
- Перед ML-7 проверить, полны ли журналы мер за март 2026 (раздел 7, holdout).
- Когда появится панель от Жени: заменить `pp.load_proto_tables` на загрузку `panel.parquet` и сверить `ds.summary`.
